# B0 — Evaluación honesta sobre conjunto de TEST independiente

Protocolo con **disciplina dev/test estricta** para una estimación de
generalización defendible frente a sobreajuste y fuga de datos.

- **Desarrollo (dev, 100 registros)**: donde se hizo TODA la selección de prompt y
  modelo. Aquí se justifica con **bootstrap** (intervalos de confianza y deltas
  pareados) por qué la configuración elegida es Exp4 (Qwen 3.5 9B + prompt v3).
- **Test (38 registros, estratificado por etiqueta, independiente)**: la
  configuración **bloqueada** Exp4 se ejecuta **una sola vez**. No se prueban otras
  configuraciones sobre el test (evita el *test farming* / comparación múltiple).

No hay entrenamiento de pesos (es prompting), por lo que "dev" = conjunto de
selección y "test" = conjunto de validación final retenido.

## 0. Setup y funciones de métrica (reutilizadas del resto del trabajo)

In [1]:
import json, sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, accuracy_score, hamming_loss, jaccard_score, precision_score, recall_score
from sklearn.preprocessing import MultiLabelBinarizer

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

LABELS = ["DIA","AAP","AAC","AAU","IIA","AAI","IAE","DUP"]   # N2, idénticas al resto
_mlb = MultiLabelBinarizer(classes=LABELS); _mlb.fit([LABELS])
SEED = 13062026
B = 2000  # remuestreos bootstrap

In [2]:
def parse_labels(v) -> list[str]:
    """Parseo tolerante (JSON, coma-separado o basura -> [])."""
    if v is None or (isinstance(v, float) and pd.isna(v)): return []
    s = str(v).strip()
    if s in ("", "nan", "None", "[]"): return []
    if s.startswith("["):
        try:
            xs = json.loads(s)
            return [str(x).strip() for x in xs if str(x).strip()] if isinstance(xs, list) else []
        except json.JSONDecodeError:
            return []
    return [x.strip().strip(chr(34)) for x in s.split(",") if x.strip()]

def parse_bool(v) -> bool:
    return v if isinstance(v, bool) else str(v).strip().lower() == "true"

def alinear(df_gt, df_pred):
    """Une GT y predicciones por descripción y devuelve arrays binarios."""
    m = df_gt[["description","is_relevant_gt","procedures_gt"]].merge(
        df_pred[["description","is_relevant_pred","procedures_pred"]], on="description", how="left")
    rel_gt = m["is_relevant_gt"].map(parse_bool).to_numpy()
    rel_pd = m["is_relevant_pred"].map(parse_bool).to_numpy()
    Y  = _mlb.transform([set(parse_labels(v)) & set(LABELS) for v in m["procedures_gt"]])
    Yh = _mlb.transform([set(parse_labels(v)) & set(LABELS) for v in m["procedures_pred"]])
    return rel_gt, rel_pd, Y, Yh, m

def metricas(idx, rel_gt, rel_pd, Y, Yh):
    return dict(
        rel_f1  = f1_score(rel_gt[idx], rel_pd[idx], zero_division=0),
        micro_f1= f1_score(Y[idx], Yh[idx], average="micro", zero_division=0),
        macro_f1= f1_score(Y[idx], Yh[idx], average="macro", zero_division=0),
        hamming = hamming_loss(Y[idx], Yh[idx]),
        jaccard = jaccard_score(Y[idx], Yh[idx], average="samples", zero_division=0),
        subset  = accuracy_score(Y[idx], Yh[idx]),
    )

def ci_bootstrap(rel_gt, rel_pd, Y, Yh, claves, B=B, seed=SEED):
    """IC95% percentil por bootstrap sobre las filas."""
    n = len(rel_gt); rng = np.random.default_rng(seed)
    idxs = [rng.integers(0, n, n) for _ in range(B)]
    full = metricas(np.arange(n), rel_gt, rel_pd, Y, Yh)
    out = {}
    for k in claves:
        vals = np.array([metricas(ix, rel_gt, rel_pd, Y, Yh)[k] for ix in idxs])
        out[k] = (full[k], *np.percentile(vals, [2.5, 97.5]))
    return out

## 1. Selección de la configuración SOBRE DEV (bootstrap)

Se evalúan las predicciones ya existentes de los 6 experimentos de desarrollo
(`results/exp1..exp6*.csv`, sin re-ejecutar el LLM). El bootstrap da intervalos de
confianza y deltas pareados que justifican la elección con respaldo estadístico.

In [3]:
EXPS_DEV = [
    ("Exp1 Baseline",  "Qwen 3.5 9B", "v1", "exp1_baseline_qwen9b.csv"),
    ("Exp2 v2",        "Qwen 3.5 9B", "v2", "exp2_promptv2_qwen9b.csv"),
    ("Exp3 v2+N1",     "Qwen 3.5 9B", "v2", "exp3_promptv2_n1_qwen9b.csv"),
    ("Exp4 v3 (REF)",  "Qwen 3.5 9B", "v3", "exp4_fewshot_qwen9b.csv"),
    ("Exp5 Gemma v2",  "Gemma 4 4B",  "v2", "exp5_promptv2_gemma4b.csv"),
    ("Exp6 v4",        "Qwen 3.5 9B", "v4", "exp6_promptv4_qwen9b.csv"),
]
gt100 = pd.read_csv(ROOT / "data/ground_truth/ground_truth_100_anotado.csv")
dev = {n: alinear(gt100, pd.read_csv(ROOT / "results" / f)) for n,_,_,f in EXPS_DEV}

filas = []
for n,modelo,ver,_ in EXPS_DEV:
    rg,rp,Y,Yh,_ = dev[n]
    ci = ci_bootstrap(rg,rp,Y,Yh, ["rel_f1","micro_f1","macro_f1","subset"])
    fmt = lambda t: f"{t[0]:.3f} [{t[1]:.2f}–{t[2]:.2f}]"
    filas.append({"Experimento":n,"Modelo":modelo,"Prompt":ver,
                  "rel F1":fmt(ci["rel_f1"]),"Micro F1":fmt(ci["micro_f1"]),
                  "Macro F1":fmt(ci["macro_f1"]),"Subset Acc":fmt(ci["subset"])})
df_dev = pd.DataFrame(filas)
print("IC95% bootstrap sobre dev (n=100, B=2000):")
display(df_dev) if "display" in dir() else print(df_dev.to_string(index=False))

IC95% bootstrap sobre dev (n=100, B=2000):
  Experimento      Modelo Prompt            rel F1          Micro F1          Macro F1        Subset Acc
Exp1 Baseline Qwen 3.5 9B     v1 0.887 [0.83–0.94] 0.891 [0.83–0.94] 0.795 [0.69–0.87] 0.800 [0.72–0.88]
      Exp2 v2 Qwen 3.5 9B     v2 0.993 [0.98–1.00] 0.972 [0.94–1.00] 0.974 [0.95–0.99] 0.950 [0.91–0.99]
   Exp3 v2+N1 Qwen 3.5 9B     v2 0.993 [0.98–1.00] 0.980 [0.96–1.00] 0.967 [0.92–0.99] 0.950 [0.90–0.99]
Exp4 v3 (REF) Qwen 3.5 9B     v3 1.000 [1.00–1.00] 0.984 [0.96–1.00] 0.975 [0.93–1.00] 0.970 [0.93–1.00]
Exp5 Gemma v2  Gemma 4 4B     v2 0.993 [0.98–1.00] 0.984 [0.96–1.00] 0.974 [0.93–1.00] 0.970 [0.93–1.00]
      Exp6 v4 Qwen 3.5 9B     v4 1.000 [1.00–1.00] 0.973 [0.94–1.00] 0.966 [0.92–1.00] 0.960 [0.92–0.99]


In [4]:
# Deltas pareados Exp4 - X (mismo remuestreo aplicado a ambos)
def deltas_pareados(ref, otros, claves=("macro_f1","micro_f1"), B=B, seed=SEED):
    rg4,rp4,Y4,Yh4,_ = dev[ref]; n=len(rg4); rng=np.random.default_rng(seed)
    idxs=[rng.integers(0,n,n) for _ in range(B)]; out=[]
    for o in otros:
        rgo,rpo,Yo,Yho,_ = dev[o]
        for k in claves:
            d=np.array([metricas(ix,rg4,rp4,Y4,Yh4)[k]-metricas(ix,rgo,rpo,Yo,Yho)[k] for ix in idxs])
            lo,hi=np.percentile(d,[2.5,97.5])
            out.append({"Comparación":f"Exp4 − {o}","Métrica":k,
                        "Δ":round(d.mean(),3),"IC95%":f"[{lo:+.2f}, {hi:+.2f}]","P(Δ>0)":round((d>0).mean(),2)})
    return pd.DataFrame(out)
df_deltas = deltas_pareados("Exp4 v3 (REF)", ["Exp1 Baseline","Exp6 v4","Exp5 Gemma v2"])
print(df_deltas.to_string(index=False))

         Comparación  Métrica     Δ          IC95%  P(Δ>0)
Exp4 − Exp1 Baseline macro_f1 0.188 [+0.11, +0.28]    1.00
Exp4 − Exp1 Baseline micro_f1 0.094 [+0.05, +0.15]    1.00
      Exp4 − Exp6 v4 macro_f1 0.009 [-0.02, +0.04]    0.75
      Exp4 − Exp6 v4 micro_f1 0.012 [-0.00, +0.03]    0.87
Exp4 − Exp5 Gemma v2 macro_f1 0.000 [-0.02, +0.02]    0.48
Exp4 − Exp5 Gemma v2 micro_f1 0.000 [-0.01, +0.01]    0.36


**Lectura**: la mejora de Exp4 sobre el baseline (Exp1) es grande y significativa
(IC del delta no cruza 0, P≈1). En cambio Exp4 vs Exp6 (v4) y, sobre todo, Exp4
(Qwen) vs Exp5 (Gemma) caen dentro del ruido del bootstrap (IC del delta incluye
0): con n=100 esas diferencias **no son estadísticamente distinguibles**. Se fija
Exp4 (Qwen+v3) como configuración de referencia —es la mejor puntualmente y empata
con las alternativas— y se reconoce explícitamente que su ventaja sobre Gemma/v4
no es significativa. Las decisiones de selección terminan aquí: el test no se usa
para elegir.

## 2. Evaluación de la config bloqueada sobre TEST (una sola vez)

Exp4 (Qwen 3.5 9B + v3) se ejecuta sobre el test independiente con
`run_experiment` (LM Studio). El CSV de predicciones se reutiliza si ya existe.

In [5]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from clasificador.agent import build_agent, run_experiment
from clasificador.schema_B0 import ClassifierOutput
from clasificador.prompts_B0 import PROMPT_REGISTRY
import asyncio

PATH_TEST_GT  = ROOT / "data/ground_truth/test_b0_para_anotado.csv"
PATH_TEST_PRED= ROOT / "results/test_b0_exp4_qwen9b.csv"

test_gt = pd.read_csv(PATH_TEST_GT)
test_gt = test_gt[test_gt["is_relevant_gt"].notna()].copy()

if not PATH_TEST_PRED.exists():
    model = OpenAIChatModel("qwen/qwen3.5-9b",
        provider=OpenAIProvider(base_url="http://localhost:1234/v1", api_key="lm-studio"))
    agent = build_agent(model, "v3", output_type=ClassifierOutput, prompt_registry=PROMPT_REGISTRY)
    asyncio.run(run_experiment(agent, test_gt, use_n1_context=False, concurrency=1,
                output_path=str(PATH_TEST_PRED), desc="TEST B0 Exp4 Qwen+v3"))
test_pred = pd.read_csv(PATH_TEST_PRED)
print(f"Predicciones de test: {len(test_pred)} filas")

Predicciones de test: 38 filas


In [6]:
rg,rp,Y,Yh,m_test = alinear(test_gt, test_pred)
ci_test = ci_bootstrap(rg,rp,Y,Yh, ["rel_f1","micro_f1","macro_f1","jaccard","subset"])
full_test = metricas(np.arange(len(rg)), rg,rp,Y,Yh)

print(f"=== TEST (n={len(rg)}) · Exp4 Qwen+v3 ===")
for k,lab in [("rel_f1","F1 relevancia"),("micro_f1","Micro F1"),("macro_f1","Macro F1"),
              ("jaccard","Jaccard"),("subset","Subset Acc")]:
    p,lo,hi = ci_test[k]
    print(f"  {lab:<15} {p:.3f}  IC95%[{lo:.2f}–{hi:.2f}]")
print(f"  Hamming Loss    {full_test['hamming']:.4f}")

# Comparación dev vs test para Exp4 (cuantifica el optimismo)
rg4,rp4,Y4,Yh4,_ = dev["Exp4 v3 (REF)"]
ci_dev4 = ci_bootstrap(rg4,rp4,Y4,Yh4, ["rel_f1","micro_f1","macro_f1","subset"])
print("\n=== Optimismo dev → test (Exp4) ===")
print(f"{'Métrica':<12}{'dev (n=100)':>22}{'test (n=38)':>22}")
for k,lab in [("rel_f1","rel F1"),("micro_f1","Micro F1"),("macro_f1","Macro F1"),("subset","Subset Acc")]:
    pd_,ld,hd = ci_dev4[k]; pt,lt,ht = ci_test[k]
    print(f"{lab:<12}{f'{pd_:.3f}[{ld:.2f}-{hd:.2f}]':>22}{f'{pt:.3f}[{lt:.2f}-{ht:.2f}]':>22}")

=== TEST (n=38) · Exp4 Qwen+v3 ===
  F1 relevancia   0.983  IC95%[0.94–1.00]
  Micro F1        0.981  IC95%[0.95–1.00]
  Macro F1        0.971  IC95%[0.82–1.00]
  Jaccard         0.754  IC95%[0.61–0.89]
  Subset Acc      0.947  IC95%[0.87–1.00]
  Hamming Loss    0.0066



=== Optimismo dev → test (Exp4) ===
Métrica                dev (n=100)           test (n=38)
rel F1            1.000[1.00-1.00]      0.983[0.94-1.00]
Micro F1          0.984[0.96-1.00]      0.981[0.95-1.00]
Macro F1          0.975[0.93-1.00]      0.971[0.82-1.00]
Subset Acc        0.970[0.93-1.00]      0.947[0.87-1.00]


## 3. Métricas por etiqueta en test (con soporte)

In [7]:
sup = Y.sum(axis=0)
print(f"{'Etiqueta':<8}{'P':>7}{'R':>7}{'F1':>7}{'Sup':>6}")
for i,lab in enumerate(LABELS):
    P=precision_score(Y[:,i],Yh[:,i],zero_division=0); R=recall_score(Y[:,i],Yh[:,i],zero_division=0)
    F=f1_score(Y[:,i],Yh[:,i],zero_division=0)
    print(f"{lab:<8}{P:>7.2f}{R:>7.2f}{F:>7.2f}{int(sup[i]):>6}")
print("\nNota: con soporte de 3-14 por etiqueta, la F1 por etiqueta y la Macro F1 son")
print("de alta varianza; las cifras estables son F1 relevancia, Micro F1 y Subset Acc.")

Etiqueta      P      R     F1   Sup
DIA        1.00   0.83   0.91     6
AAP        1.00   1.00   1.00    14
AAC        1.00   1.00   1.00    14
AAU        1.00   1.00   1.00     3
IIA        1.00   1.00   1.00     3
AAI        1.00   0.75   0.86     4
IAE        1.00   1.00   1.00     3
DUP        1.00   1.00   1.00     8

Nota: con soporte de 3-14 por etiqueta, la F1 por etiqueta y la Macro F1 son
de alta varianza; las cifras estables son F1 relevancia, Micro F1 y Subset Acc.


## 4. Análisis de errores del test (pred ≠ GT)

In [8]:
def labset(v): return sorted(set(parse_labels(v)) & set(LABELS))
m_test["gt_set"]   = m_test["procedures_gt"].map(labset)
m_test["pred_set"] = m_test["procedures_pred"].map(labset)
m_test["rel_gt"]   = m_test["is_relevant_gt"].map(parse_bool)
m_test["rel_pd"]   = m_test["is_relevant_pred"].map(parse_bool)
err = m_test[[a!=b or g!=p for a,b,g,p in
              zip(m_test["gt_set"],m_test["pred_set"],m_test["rel_gt"],m_test["rel_pd"])]]
print(f"Errores de acierto exacto: {len(err)} de {len(m_test)}\n")
for _,r in err.iterrows():
    falta=sorted(set(r['gt_set'])-set(r['pred_set'])); sobra=sorted(set(r['pred_set'])-set(r['gt_set']))
    print(f"GT={r['gt_set']} relGT={r['rel_gt']} | PRED={r['pred_set']} relPD={r['rel_pd']}")
    print(f"  falta={falta} sobra={sobra}")
    print(f"  {str(r['description'])[:110]}")
    print()

Errores de acierto exacto: 2 de 38

GT=['AAI'] relGT=True | PRED=[] relPD=False
  falta=['AAI'] sobra=[]
  Impacto ambiental
– Resolución de 13 de diciembre de 2024, de la Directora General de Transición Energética y 

GT=['AAC', 'AAP', 'DIA'] relGT=True | PRED=['AAC', 'AAP'] relPD=True
  falta=['DIA'] sobra=[]
  Anuncio por el que se somete a información pública la solicitud de autorización administrativa previa, autoriz



## 5. Sesgos y limitaciones del test (declarar en la memoria)

1. **Estratificación por etiqueta**: el test sobre-representa deliberadamente las
   etiquetas N2. Las métricas son **condicionales a ese diseño**, NO la prevalencia
   real del corpus (B0 ≈ 3,4 %). No deben leerse como rendimiento poblacional.
2. **Selección por keyword**: los candidatos se eligieron por señales léxicas
   explícitas; en la práctica todos los candidatos resultaron del dominio y todos
   los negativos, negativos. El test mide bien la **asignación de etiquetas con
   señal explícita**, pero NO la capacidad de **rechazar falsos positivos léxicos**
   (publicaciones con la keyword pero fuera de dominio): no hay "negativos
   difíciles".
3. **Optimismo de desarrollo**: dev se usó para seleccionar, así que sus métricas
   son optimistas; el test corrige ese sesgo y es la cifra de generalización.
4. **Anotación de un solo anotador**: sin acuerdo inter-anotador (Cohen's κ); hay
   ruido de etiquetado no cuantificado.
5. **N pequeño (38)**: intervalos de confianza anchos; Macro F1 y F1 por etiqueta
   de etiquetas raras son inestables. Se reportan siempre con IC bootstrap.